# Full 1536 + crop 학습 probe — 다른 PC용 (<호스트명> 금지)

**배경.** `<호스트명>`(로컬 5060 Ti #1)에서 이 측정을 시도하다가 같은 날 두 번 하드웨어
레벨로 죽었다(한 번은 `nvidia-smi`가 "GPU is lost", 한 번은 PC 전체 전원 차단 — 둘 다 모델
로딩 중 VRAM이 16GB 카드의 98% 안팎까지 차오르는 순간과 겹쳤다. 150W 전력 상한을 걸어도
재현됐다). Docker 컨테이너는 물리 GPU/드라이버/전원 공급장치를 그대로 공유하므로
**컨테이너로 감싼다고 이 종류의 크래시가 막히지 않는다** — 이 노트북이 확인하려는 건
그게 이 PC 개별 하드웨어 문제인지, 카드/설정 공통 문제인지다.

**목적.** 09/17 세션 이후 미측정으로 남아 있던 두 값을 다른 5060 Ti PC에서 확보한다:
1. 학습 Full 1536 (1,728 visual tokens로 추정 — VRAM이 Full 1024를 초과할 것으로 예상되지만 미측정)
2. 학습 crops=4 @768 (베어메탈에서는 900초 TIMEOUT — sysmem 폴백, OOM 아님)

**실행 전 필수 확인 (아래 첫 코드 셀이 강제한다):**
- `<호스트명>`가 **아닌** PC에서 실행할 것 — 첫 셀에서 호스트명을 직접 입력받아 대조한다
- 이 저장소의 `docker compose up --build -d`로 뜬 `ai2-challenge-lab` 컨테이너 안에서 Jupyter를 열 것
  (Docker 환경이 베어메탈과 거의 동일하다는 건 <호스트명>에서 이미 검증됨 —
  추론 14.039 vs 14.0148 GB, 학습 15.263 vs 15.215 GB / 2.206 vs 2.323 sec/step, 둘 다 1% 미만 drift)
- 전력 로거가 1초 간격으로 기록한다 — **PC가 꺼져도 그 직전까지는 디스크에 남는다.**
  죽었으면 `power_log_*.csv`만이라도 그대로 팀에 보내달라 — 이 PC가 버티는지 자체가 중요한 정보다

## 0. 호스트명 확인 — <호스트명>면 여기서 멈춘다

컨테이너 안 `socket.gethostname()`은 컨테이너 ID라 실제 PC를 구분 못 한다.
**이 PC의 실제 컴퓨터 이름을 직접 입력해라** (Windows에서 `hostname` 명령으로 확인).

In [ ]:
BANNED_HOST = "<호스트명>"

actual_hostname = input("이 PC의 실제 호스트명을 입력하세요 (Windows에서 `hostname` 명령 결과): ").strip()

if actual_hostname.upper() == BANNED_HOST.upper():
    raise SystemExit(
        f"\n*** 중단: {BANNED_HOST}는 오늘 두 번(하드웨어 레벨) 크래시가 확인된 PC입니다. ***\n"
        "*** 다른 PC에서 실행해주세요. ***"
    )
if not actual_hostname:
    raise SystemExit("호스트명을 입력하지 않았습니다. 위 셀을 다시 실행하세요.")

print(f"확인됨: {actual_hostname} — {BANNED_HOST}가 아니므로 계속 진행합니다.")

## 1. 환경 스냅샷

In [ ]:
import datetime, json, os, platform, socket, subprocess, sys, threading, time
from pathlib import Path

HOSTNAME = actual_hostname
START_TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
IS_DOCKER = Path("/.dockerenv").exists()
ENV = "docker_wsl2" if IS_DOCKER else "bare_windows"
SYSMEM_UNAVAILABLE_LABEL = "N/A(non-Windows container — WDDM sysmem fallback 감지 불가)"

if not IS_DOCKER:
    print("*** 경고: /.dockerenv가 없습니다 — Docker 컨테이너 안이 아닐 수 있습니다. ***")
    print("*** ai2-challenge-lab 컨테이너의 JupyterLab(기본 포트 8889)에서 열었는지 확인하세요. ***")

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print("REPO_ROOT:", REPO_ROOT)
os.chdir(REPO_ROOT)

import torch
TORCH_VERSION = torch.__version__
print("env:", ENV)
print("hostname:", HOSTNAME)
print("torch:", TORCH_VERSION)
print("cuda available:", torch.cuda.is_available())

def nvidia_smi_query(fields):
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=10,
    ).stdout.strip()
    return [line.split(", ") for line in out.splitlines()]

for row in nvidia_smi_query("name,driver_version,memory.total,memory.free,power.limit"):
    print("gpu:", row[0], "| driver:", row[1], "| total_mib:", row[2], "| free_mib:", row[3], "| power_limit_w:", row[4])

GPU_NAME = nvidia_smi_query("name")[0][0]
GPU_UUID = nvidia_smi_query("uuid")[0][0]
GPU_UUID_SHORT = GPU_UUID.replace("GPU-", "")[:8]
print("gpu_uuid:", GPU_UUID, "| short:", GPU_UUID_SHORT)

**왜 UUID까지 남기나.** 교육장 PC는 hostname이 전부 `<호스트명>`로 동일해서
hostname만으로는 어느 PC(어느 카드)에서 나온 결과인지 구분이 안 된다. `input()`으로
받는 호스트명은 오타 방지용 1차 확인일 뿐이고, **GPU UUID 앞 8자리가 실제 유일 식별자**다
(참고: 박건순 PC는 `8252eea8`). 파일명·결과 CSV 모두 UUID로 구분한다.

## 2. 전력 상한 확인 — 150W 권장

`<호스트명>`에서 두 크래시 다 150W 상한을 걸어도 재현됐지만, 상한이 아예 없는 것보다는
안전하다. **이 PC에서도 관리자 권한 PowerShell로 `nvidia-smi -pl 150`을 먼저 걸고 오는 걸 권장한다**
(재부팅하면 풀리므로 매번 다시 걸어야 한다). 아래 셀은 현재 상한만 보여준다 — 강제하지 않는다.

In [ ]:
limit = nvidia_smi_query("power.limit")[0][0]
print(f"현재 power.limit: {limit} W")
if float(limit) > 150:
    print("*** 150W보다 높습니다. 가능하면 관리자 PowerShell에서 `nvidia-smi -pl 150` 실행 후 이 셀부터 다시 실행하세요. ***")

## 3. 전력 로거 — 1초 간격, 매 줄 flush+fsync

PC가 꺼져도 이 CSV의 마지막 줄까지는 디스크에 남는다.

In [ ]:
import csv

POWER_LOG_PATH = REPO_ROOT / f"power_log_{HOSTNAME}_{GPU_UUID_SHORT}_{ENV}_{START_TS}.csv"
POWER_LOG_FIELDS = ["timestamp", "power_draw_w", "temperature_c", "utilization_pct", "memory_used_mib"]
_power_log_stop = threading.Event()

def _power_log_loop():
    with open(POWER_LOG_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if f.tell() == 0:
            writer.writerow(POWER_LOG_FIELDS)
            f.flush(); os.fsync(f.fileno())
        while not _power_log_stop.is_set():
            try:
                rows = nvidia_smi_query("power.draw,temperature.gpu,utilization.gpu,memory.used")
                ts = datetime.datetime.now().isoformat()
                for row in rows:
                    writer.writerow([ts, *row])
                f.flush(); os.fsync(f.fileno())
            except Exception as exc:
                print("power log 폴링 실패(계속 시도함):", exc)
            _power_log_stop.wait(1.0)

_power_log_thread = threading.Thread(target=_power_log_loop, daemon=True)
_power_log_thread.start()
print(f"전력 로거 시작: {POWER_LOG_PATH}")

## 4. 결과 CSV + 측정 함수

베어메탈/<호스트명> Docker 결과와 섞이지 않도록 파일명에 hostname+env를 넣는다.

In [ ]:
RESULTS_PATH = REPO_ROOT / f"results_{HOSTNAME}_{GPU_UUID_SHORT}_{ENV}.csv"
RESULTS_FIELDS = [
    "hostname", "gpu_uuid_short", "env", "gpu_name", "torch_version", "mode", "long_side", "crops", "crop_long_side",
    "visual_tokens", "peak_vram_torch", "peak_vram_smi", "peak_shared_mem_gb_windows",
    "sec_per_step_steady", "sec_per_step_warmup", "status", "notes",
    "fla_version", "triton_version", "gated_delta_rule_impl", "causal_conv1d_impl",
]

def _write_result_row(row):
    is_new = not RESULTS_PATH.exists()
    with open(RESULTS_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in RESULTS_FIELDS})
        f.flush(); os.fsync(f.fileno())

print(f"결과 CSV: {RESULTS_PATH}")

def _run_subprocess(args, timeout_sec=900):
    proc = subprocess.run(
        [sys.executable, "-m", "src.run", *args],
        cwd=str(REPO_ROOT), capture_output=True, text=True, timeout=timeout_sec,
    )
    return proc.returncode, (proc.stdout or "") + (proc.stderr or "")

def _classify_status(returncode, output):
    if returncode == 0:
        return "ok"
    if "OutOfMemoryError" in output or "CUDA out of memory" in output:
        return "OOM"
    return f"ERROR(exit={returncode})"

def _apply_kernel_impl(row, kernel_impl):
    if not kernel_impl:
        return
    row["fla_version"] = kernel_impl.get("fla_version")
    row["triton_version"] = kernel_impl.get("triton_version")
    row["gated_delta_rule_impl"] = kernel_impl.get("gated_delta_rule_impl")
    row["causal_conv1d_impl"] = kernel_impl.get("causal_conv1d_impl")

def measure_train(name, n_steps=12, grad_checkpointing=True, crops=0, crop_long_side=None, long_side=1024):
    args = ["train", "-c", "9b_qlora_r8_1024", "--synthetic", "-n", str(n_steps),
            "--set", f"input.long_side={long_side}",
            "--set", f"input.crops={crops}",
            "--set", f"train.grad_checkpointing={'true' if grad_checkpointing else 'false'}"]
    if crop_long_side is not None:
        args += ["--set", f"input.crop_long_side={crop_long_side}"]

    row = {"hostname": HOSTNAME, "gpu_uuid_short": GPU_UUID_SHORT, "env": ENV, "gpu_name": GPU_NAME,
           "torch_version": TORCH_VERSION, "mode": "train",
           "long_side": long_side, "crops": crops, "crop_long_side": crop_long_side}
    print(f"=== [train] {name} ===")
    t_start = time.time()
    try:
        rc, output = _run_subprocess(args)
        elapsed = time.time() - t_start
        status = _classify_status(rc, output)
        summary_path = REPO_ROOT / "runs" / "trainprobe_9b_qlora_r8_1024" / "summary.json"
        if status == "ok" and summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            probe = summary.get("train_probe", {})
            row["peak_vram_torch"] = probe.get("peak_vram_gb_torch")
            row["peak_vram_smi"] = probe.get("peak_vram_gb_nvidia_smi")
            row["sec_per_step_steady"] = probe.get("steady_state_sec_per_step_median")
            row["sec_per_step_warmup"] = probe.get("warmup_sec_per_step")
            sysmem = probe.get("peak_shared_mem_gb_windows")
            row["peak_shared_mem_gb_windows"] = SYSMEM_UNAVAILABLE_LABEL if (IS_DOCKER and sysmem is None) else sysmem
            _apply_kernel_impl(row, probe.get("kernel_impl"))
        elif IS_DOCKER:
            row["peak_shared_mem_gb_windows"] = SYSMEM_UNAVAILABLE_LABEL
        row["status"] = status
        if status != "ok":
            row["notes"] = output[-400:]
        print(f"  -> {status}  elapsed={elapsed:.1f}s  peak_vram_torch={row.get('peak_vram_torch')}  "
              f"steady sec/step={row.get('sec_per_step_steady')}")
        if status != "ok":
            print(output[-2000:])
    except subprocess.TimeoutExpired as exc:
        elapsed = time.time() - t_start
        row["status"] = "TIMEOUT"
        # capture_output=True면 타임아웃에도 그때까지 캡처된 출력이 exc.stdout/stderr에 남는다
        partial = ((exc.stdout or "") + (exc.stderr or "")) if hasattr(exc, "stdout") else ""
        row["notes"] = partial[-400:] if partial else "(서브프로세스 출력 없음)"
        if IS_DOCKER:
            row["peak_shared_mem_gb_windows"] = SYSMEM_UNAVAILABLE_LABEL
        print(f"  -> TIMEOUT (elapsed={elapsed:.1f}s) — 900초를 꽉 채웠으면 sysmem 폴백, 훨씬 짧고 OOM 메시지가 있었으면 진짜 OOM입니다")
    except Exception as exc:
        row["status"] = f"ERROR({type(exc).__name__})"
        row["notes"] = str(exc)[-400:]
        print("  -> ERROR:", exc)
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    _write_result_row(row)
    return row

## 5. 대조군 — 학습 Full 1024 (먼저 돌린다, 3분 내외)

**이 셀을 건너뛰지 마라.** 기준값: 베어메탈 15.215 GB / 2.323 sec/step,
<호스트명> Docker 15.263 GB / 2.206 sec/step. **이 PC의 값이 이 범위(대략 ±20%)를
크게 벗어나면 이 PC 자체가 다른 상태(다른 드라이버, 다른 카드 상태, 백그라운드 GPU 점유 등)라는
뜻이므로 아래 6절(Full 1536) 결과를 곧이곧대로 믿으면 안 된다.** 대조군 없이 1536만 돌리면
실패했을 때 "1536이라서 실패했는지" "이 PC라서 실패했는지"를 가릴 수 없다.

In [ ]:
measure_train("대조군: Full 1024", grad_checkpointing=True, crops=0, long_side=1024)

## 6. (c) 학습 Full 1536 — ★ 이번 세션의 핵심 미측정값

베어메탈 Full 1024 기준: 15.215 GB / 2.323 sec/step. <호스트명> Docker Full 1024: 15.263 GB / 2.206 sec/step.
Full 1536은 1,728 visual tokens로 추정되어 VRAM이 이 기준을 넘을 가능성이 높다 — 16GB 카드 한도 안에
들어가는지가 관건이다. **최대 15분 걸릴 수 있고, 여기서 크래시가 나면 그 자체가 중요한 정보다** —
죽어도 전력 로그는 그 직전까지 남아 있으니 파일만 챙겨서 보내면 된다.

In [ ]:
measure_train("Full 1536 (핵심 미측정값)", grad_checkpointing=True, crops=0, long_side=1536)

## 7. (d) crops=4 @768 — 베어메탈은 TIMEOUT(sysmem 폴백)이었다

베어메탈에서 900.6~901.5초로 15분을 꽉 채운 TIMEOUT이었다(OOM 아님). 여기서도 TIMEOUT이면
Docker/Linux 환경에서도 같은 폴백 패턴이 있다는 뜻이고, 짧고 굵게 OOM으로 끝나면 환경 차이로
폴백 자체가 사라졌다는 뜻이다 — 의미가 반대이니 상태를 정확히 구분해서 보고할 것.

In [ ]:
measure_train("crops=4, crop_long_side=768", grad_checkpointing=True, crops=4, crop_long_side=768, long_side=1024)

## 8. 마무리

In [ ]:
_power_log_stop.set()
_power_log_thread.join(timeout=5)
print("전력 로거 정지.")
print("\n결과 파일:", RESULTS_PATH.resolve())
print("전력 로그:", POWER_LOG_PATH.resolve())
print("\n이 두 CSV를 그대로 팀에 보내주세요. 중간에 멈췄어도 전력 로그는 그 직전까지 남아 있습니다.")